### Import libraries

In [1]:
import os
import pickle
from methyldl.deconvolution.xgbdeconvolver import *
from tqdm import tqdm
from collections import defaultdict
import torch.nn as nn
from copy import deepcopy
from methyldl.deconvolution.deep_deconvolvers.training import train_matrix_deconvolver
import os.path as Path
from torch.utils.data import DataLoader, TensorDataset
from methyldl.deconvolution.least_squares_deconvolvers import (
    NNLSDeconvolver,
    PSLSDeconvolver,
)
from methyldl.deconvolution.evaluation import compute_deconvolution_metrics
from methyldl.deconvolution.linear_calibrator import LinearCalibrator

### Configuring target model and extracting computed average scores

In [15]:
from edautils import *

reads_data_path = "/home/luna.kuleuven.be/u0169940/Data/Loyfer/SoftLabelsForRRBSsplits_205files_pooled_Jaccard_hg38_mincpg_4_minlen_10_d041/"
dmr_label_column = "dmr_ctype_label"
mincpg_pointer = reads_data_path.find("mincpg_")
# classifier_model_path = f"../Tutorials/methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_hg38_{dmr_label_column}_{reads_data_path[mincpg_pointer:]}"
classifier_model_path = "/home/luna.kuleuven.be/u0169940/Repos/methyldl/Experiments/RRBS/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_d041_postfiltered_min_length_50_soft_labels_pooled_jakkard/pseudobulk"
mincpg = int(reads_data_path[mincpg_pointer + 7 : mincpg_pointer + 8])
features_encoding = "extracted_numpy"
n_cell_types = num_dmr_groups = 39
n_pred_classes = 39 if "soft_labels" in classifier_model_path else 40

In [16]:
features_file = [
    x for x in os.listdir(classifier_model_path) if "features_cutoff" in x
][0]

In [17]:
df = np.load(Path.join(classifier_model_path, features_file))

In [18]:
# features_test = df["features_test"]
features_train = df["features_train"]
features_valid = df["features_valid"]
target_proportions = df["proportions"]

### Defining deconvolvers

In [19]:
n_features = 156

In [20]:
swn = nn.Sequential(
    nn.Linear(n_features, 1024),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(1024, 39),
    nn.Softmax(dim=-1),
)

mlp = nn.Sequential(
    nn.Linear(n_features, 512),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(512, 256),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(256, n_features),
    nn.GELU(),
    nn.Dropout(0.1),
    nn.Linear(n_features, 39),
    nn.Softmax(dim=-1),
)

xgb_config = XGBDeconvolverConfig(
    n_estimators=500,
    max_depth=15,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=1,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    early_stopping_rounds=100,
    random_state=42,
)

xgb = XGBoostDeconvolver(
    config=xgb_config,
    output_transform="clip_normalize",
    n_dmr_groups=num_dmr_groups,
    n_pred_classes=n_pred_classes,
    n_cell_types=n_cell_types,
    with_reject_features=False,
    process_inputs=False,
)

nnls = NNLSDeconvolver()
psls = PSLSDeconvolver()

### Loading deconvolvers weights

In [21]:
device = "cuda"

In [22]:
swn.to(device=device)
swn.load_state_dict(
    torch.load(
        Path.join(classifier_model_path, "swn_best_deconvolver.pt"), weights_only=True
    )
)
swn.eval()
mlp.to(device=device)
mlp.load_state_dict(
    torch.load(
        Path.join(classifier_model_path, "mlp_best_deconvolver.pt"), weights_only=True
    )
)
mlp.eval()
xgb = xgb.load(Path.join(classifier_model_path, "xgb_deconvolver.joblib"))
nnls = nnls.load(Path.join(classifier_model_path, "nnls_deconvolver.joblib"))
psls = psls.load(Path.join(classifier_model_path, "psls_deconvolver.joblib"))

### Generating predictions

In [23]:
results = defaultdict(tuple)
model_tripples = [
    (swn, "swn", "nn"),
    (mlp, "mlp", "nn"),
    (xgb, "xgb", "xgb"),
    (nnls, "nnls", "ls"),
    (psls, "psls", "ls"),
]

In [24]:
def infer_multiple_deconvolvers(features, target_proportions, model_tripples):
    test_loader = DataLoader(
        TensorDataset(
            torch.FloatTensor(features), torch.FloatTensor(target_proportions)
        ),
        batch_size=2000,
    )
    results = defaultdict(tuple)
    for model, model_name, model_type in model_tripples:
        all_preds = []
        all_targets = target_proportions
        if model_type == "nn":
            all_targets = []
            with torch.no_grad():
                for X, y in test_loader:
                    X, y = X.to(device), y.to(device)
                    pred = model(X)
                    all_preds.append(pred.cpu())
                    all_targets.append(y.cpu())
            # Compute all metrics
            all_preds = torch.cat(all_preds, dim=0).numpy()
            all_targets = torch.cat(all_targets, dim=0)
            all_targets = all_targets.numpy()
        elif model_type == "xgb":
            all_preds = model._predict_raw(features)
            all_preds = model._transform_output(all_preds)
        elif model_type == "ls":
            if "nnls" in model_name:
                all_preds, _, _ = model.predict(features, n_workers=1)
            elif "psls" in model_name:
                all_preds = model.predict(features, n_workers=2)
        else:
            pass
        metrics = compute_deconvolution_metrics(all_preds, all_targets)
        all_preds = np.round(all_preds, 4)

        results[model_name] = (all_targets, all_preds, metrics)

    return results

In [ ]:
results_test = infer_multiple_deconvolvers(
    features=features_test,
    target_proportions=target_proportions,
    model_tripples=model_tripples,
)

In [27]:
import pandas as pd


def results_to_dataframe(results: dict, class_names: list = None) -> pd.DataFrame:
    """
    Convert the results dict into a DataFrame matching the supplementary table columns.

    Expected results structure:
        results[model_name] = (all_targets, all_preds, metrics_dict)

    model_name convention assumed (adjust parsing as needed):
        e.g. "hard_rej__dsimir__dirichlet__xgb__linear"
              labeling__classifier__clf_calib__deconvolver__dec_calib
    """
    rows = []
    for model_name, (targets, preds, metrics) in results.items():
        # --- parse model_name into table keys ---
        parts = model_name.split("__")
        if len(parts) == 5:
            labeling, classifier, clf_calib, deconvolver, dec_calib = parts
        elif len(parts) == 3:
            # UXM case: e.g. "uxm__uxm__linear"
            labeling, deconvolver, dec_calib = parts
            classifier = "---"
            clf_calib = "---"
        else:
            # fallback: store raw name, fill manually
            labeling = ""
            classifier = ""
            clf_calib = ""
            deconvolver = model_name
            dec_calib = ""

        worst_class = metrics["worst_class_name"]
        if class_names is not None and isinstance(worst_class, int):
            worst_class = class_names[worst_class]

        rows.append(
            {
                # "Labeling": labeling,
                # "Classifier": classifier,
                # "Clf. Calib.": clf_calib,
                "Deconvolver": deconvolver,
                # "Dec. Calib.": dec_calib,
                "R2": (
                    1.0 - metrics["mse"] / targets.var()
                    if hasattr(targets, "var")
                    else None
                ),
                "LoA": f"[{metrics['loa_lower']:.4f}, {metrics['loa_upper']:.4f}]",
                "LoA width": round(metrics["loa_width"], 4),
                "LoA (worst)": f"[{metrics['worst_class_loa_lower']:.4f}, {metrics['worst_class_loa_upper']:.4f}]",
                "LoA width (worst)": round(metrics["worst_class_loa_width"], 4),
                "Worst class": worst_class,
                "MSE": round(metrics["mse"], 6),
                "MAE": round(metrics["mae"], 6),
                "KL": round(metrics["kl"], 6),
                "Cosine Sim": round(metrics["cosine_sim"], 6),
            }
        )

    df = pd.DataFrame(rows)

    # Sort to match table grouping order
    df = df.sort_values(by=["Deconvolver"]).reset_index(drop=True)

    return df

In [ ]:
results_to_dataframe(results_test)

### Infering linear callibrators

In [ ]:
results_test.keys()

In [ ]:
def apply_fitted_callibration(results_test):
    results = defaultdict(tuple)
    for model_name in results_test.keys():
        test_preds = results_test[model_name][0]
        test_target = results_test[model_name][1]

        calibrator = LinearCalibrator()
        calibrator.load_calibration_parameters(
            Path.join(classifier_model_path, f"{model_name}_linear_calibrator.npz")
        )
        calibrated_test_pred, _ = calibrator.predict(test_preds)

        metrics = compute_deconvolution_metrics(calibrated_test_pred, test_target)
        calibrated_test_pred = np.round(calibrated_test_pred, 4)

        results[model_name] = (test_target, calibrated_test_pred, metrics)

    return results

In [ ]:
results_calibrated = apply_fitted_callibration(results_test)

In [ ]:
results_to_dataframe(results_calibrated)

### Fitting linear callibrators

In [25]:
results_valid = infer_multiple_deconvolvers(
    features=features_valid,
    target_proportions=target_proportions,
    model_tripples=model_tripples,
)

Predicting with PSLS in parallel: 100%|██████████| 1000/1000 [02:16<00:00,  7.35it/s]


In [28]:
results_to_dataframe(results_valid)

,Deconvolver,R2,LoA,LoA width,LoA (worst),LoA width (worst),Worst class,MSE,MAE,KL,Cosine Sim
0,mlp,0.971299,"[-0.0302, 0.0302]",0.0604,"[-0.0620, 0.1037]",0.1657,11,0.000238,0.003561,0.069975,0.984498
1,nnls,0.984907,"[-0.0219, 0.0219]",0.0438,"[-0.0681, 0.0600]",0.1281,11,0.000125,0.003194,0.076115,0.993617
2,psls,0.983661,"[-0.0228, 0.0228]",0.0456,"[-0.0719, 0.0616]",0.1335,11,0.000135,0.003233,0.076135,0.992922
3,swn,0.975181,"[-0.0281, 0.0281]",0.0562,"[-0.0873, 0.1204]",0.2078,11,0.000206,0.003405,0.057047,0.986290
4,xgb,0.958051,"[-0.0365, 0.0365]",0.0731,"[-0.1339, 0.1474]",0.2813,11,0.000347,0.004240,0.077863,0.982577


In [29]:
def apply_callibration(results_valid, results_test, save_calibrators=False):
    results = defaultdict(tuple)
    for model_name in results_valid.keys():
        valid_preds = results_valid[model_name][0]
        val_target = results_valid[model_name][1]

        test_preds = results_test[model_name][0]
        test_target = results_test[model_name][1]

        calibrator = LinearCalibrator()
        calibrator.fit(valid_preds, val_target)
        calibrated_test_pred, _ = calibrator.predict(test_preds)

        metrics = compute_deconvolution_metrics(calibrated_test_pred, test_target)
        calibrated_test_pred = np.round(calibrated_test_pred, 4)

        results[model_name] = (test_target, calibrated_test_pred, metrics)

        if save_calibrators:
            calibrator.save_calibration_parameters(
                f"{classifier_model_path}/{model_name}_linear_calibrator.npz"
            )

    return results

In [ ]:
calibrated_test_results = apply_callibration(
    results_valid, results_test, save_calibrators=True
)

In [32]:
results_to_dataframe(calibrated_test_results)

,Deconvolver,R2,LoA,LoA width,LoA (worst),LoA width (worst),Worst class,MSE,MAE,KL,Cosine Sim
0,mlp,0.977228,"[-0.0262, 0.0262]",0.0524,"[-0.0823, 0.0813]",0.1636,11,0.000179,0.003779,0.097195,0.988208
1,nnls,0.991746,"[-0.0154, 0.0154]",0.0309,"[-0.0422, 0.0456]",0.0878,11,0.000062,0.003058,0.087987,0.996766
2,psls,0.991157,"[-0.0160, 0.0160]",0.0320,"[-0.0425, 0.0462]",0.0887,11,0.000067,0.003070,0.087735,0.996648
3,swn,0.980766,"[-0.0244, 0.0244]",0.0488,"[-0.1026, 0.1013]",0.2039,11,0.000155,0.003283,0.109167,0.989614
4,xgb,0.966188,"[-0.0306, 0.0306]",0.0612,"[-0.1230, 0.1270]",0.2501,11,0.000244,0.005109,0.151195,0.985607
